In [0]:
%run ../config_setup

# Silver 01 — Machine Master (SCD2)

RDS CDC se machine_master events uthata hai, latest `_lsn` per machine nikaalta hai, aur env-configured `machine_master` silver table me PySpark SCD2 apply karta hai.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, current_timestamp, expr, lit
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
TARGET_TABLE = f"{SILVER}.machine_master"

# Bronze se saare zaroori columns extract diye (manufacture_year ke saath)
df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "machine_master").select(
    col("_lsn"), col("_change_type"),
    get_json_object("after", "$.machine_id").alias("machine_id"),
    get_json_object("after", "$.model_id").alias("model_id"),
    get_json_object("after", "$.owner_id").alias("owner_id"),
    get_json_object("after", "$.site_id").alias("site_id"),
    get_json_object("after", "$.status").alias("status"),
    get_json_object("after", "$.manufacture_year").cast("int").alias("manufacture_year")
)

def upsert_to_delta(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    
    latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by machine_id order by _lsn desc)")).filter("rn = 1").drop("rn")

    if not spark.catalog.tableExists(TARGET_TABLE):
        (latest.filter(col("_change_type") != "delete")
               .drop("_lsn", "_change_type")
               .withColumn("__START_AT", current_timestamp())
               .withColumn("__END_AT", lit(None).cast("timestamp"))
               .withColumn("__CURRENT", lit(True))
               .write.format("delta").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE))
    else:
        target = DeltaTable.forName(spark, TARGET_TABLE)
        
        # Delete
        deletes = latest.filter(col("_change_type") == "delete")
        target.alias("t").merge(deletes.alias("s"), "t.machine_id = s.machine_id AND t.__CURRENT = true") \
              .whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
              
        # Update/Insert
        upserts = latest.filter(col("_change_type") != "delete")
        target.alias("t").merge(upserts.alias("s"), "t.machine_id = s.machine_id AND t.__CURRENT = true") \
              .whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
              
        (upserts.drop("_lsn", "_change_type")
                .withColumn("__START_AT", current_timestamp())
                .withColumn("__END_AT", lit(None).cast("timestamp"))
                .withColumn("__CURRENT", lit(True))
                .write.format("delta").mode("append").saveAsTable(TARGET_TABLE))

df.writeStream.foreachBatch(upsert_to_delta).option("checkpointLocation", f"{CHECKPOINTS}/silver_machine_master").trigger(availableNow=True).start().awaitTermination()